# 01 - Data Exploration

Explore reference and production datasets.

- Load reference and production data
- Inspect distributions, data types, missing values
- Compare feature statistics
- Identify potential data drift candidates

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

PROJECT_ROOT = Path.cwd()
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\ACS\Desktop\ML-model-monitoring-dashboard\notebooks


In [3]:
from src.data.data_loader import load_reference_data, load_production_data, load_config

config = load_config()
reference = load_reference_data(config)
production = load_production_data(config)

print(f"Reference shape: {reference.shape}")
print(f"Production shape: {production.shape}")
print()
print("Reference columns:", list(reference.columns))
print("Production columns:", list(production.columns))

ModuleNotFoundError: No module named 'src'

In [ ]:
display(reference.head())
display(production.head())

In [ ]:
print("=== REFERENCE ===")
print(reference.dtypes)
print()
print("Missing values:")
print(reference.isnull().sum())
print()
print("=== PRODUCTION ===")
print(production.dtypes)
print()
print("Missing values:")
print(production.isnull().sum())

In [4]:
fig_ref = px.histogram(reference, x="target", title="Reference Target Distribution", labels={"target": "Target Class"}, color_discrete_sequence=["#2E86AB"])
fig_prod = px.histogram(production, x="target", title="Production Target Distribution", labels={"target": "Target Class"}, color_discrete_sequence=["#F18F01"])

fig_ref.show()
fig_prod.show()

NameError: name 'reference' is not defined

In [ ]:
if "category" in reference.columns:
    fig_cat_ref = px.histogram(reference, x="category", title="Reference Category Distribution", color_discrete_sequence=["#2E86AB"])
    fig_cat_prod = px.histogram(production, x="category", title="Production Category Distribution", color_discrete_sequence=["#F18F01"])
    fig_cat_ref.show()
    fig_cat_prod.show()

    print("Reference category proportions:")
    print(reference["category"].value_counts(normalize=True).sort_index())
    print()
    print("Production category proportions:")
    print(production["category"].value_counts(normalize=True).sort_index())

In [ ]:
numeric_cols = reference.select_dtypes(include=["number"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != "target"]

comparison = pd.DataFrame({
    "feature": numeric_cols,
    "ref_mean": [reference[c].mean() for c in numeric_cols],
    "prod_mean": [production[c].mean() for c in numeric_cols],
    "ref_std": [reference[c].std() for c in numeric_cols],
    "prod_std": [production[c].std() for c in numeric_cols],
})
comparison["mean_diff_pct"] = ((comparison["prod_mean"] - comparison["ref_mean"]) / comparison["ref_mean"].abs() * 100).round(2)

display(comparison)

In [ ]:
import plotly.graph_objects as go

for col in numeric_cols[:4]:
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=reference[col].dropna(), name="Reference", opacity=0.7, marker_color="#2E86AB", nbinsx=30))
    fig.add_trace(go.Histogram(x=production[col].dropna(), name="Production", opacity=0.7, marker_color="#F18F01", nbinsx=30))
    fig.update_layout(title=f"Distribution: {col}", barmode="overlay", xaxis_title=col, yaxis_title="Count", template="plotly_white")
    fig.show()

## Key Observations

- The production dataset includes a `prediction_id` column for tracking individual predictions.
- Categorical feature `category` shows a distribution shift between reference and production.
- Several numeric features show mean shifts, indicating potential covariate drift.
- These observations justify running data drift detection in the monitoring pipeline.